# DFK Model → GGUF Converter
Jalankan **satu per satu**. Jangan skip cell apapun.

**Fix:** model ini punya tensor prefix `model.model.` (dari PEFT training). Cell 4.5 mempatch converter.

In [ ]:
# Cell 1: Install dependency
import subprocess, sys, os
subprocess.run([sys.executable,'-m','pip','install','-q',
    'huggingface_hub','transformers','sentencepiece','gguf','numpy'], check=True)
print('Done')

In [ ]:
# Cell 2: Clone llama.cpp
LLAMA_DIR  = '/content/llama.cpp'
MODEL_DIR  = '/content/dfk_model'
OUTPUT_F16 = '/content/model-f16.gguf'
OUTPUT_Q4  = '/content/model-q4_k_m.gguf'

if not os.path.exists(LLAMA_DIR):
    subprocess.run(['git','clone','--depth=1',
        'https://github.com/ggerganov/llama.cpp', LLAMA_DIR], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q',
        '-r', f'{LLAMA_DIR}/requirements.txt'], check=True)
print('llama.cpp ready')

In [ ]:
# Cell 3: Download model (~35 GB)
from huggingface_hub import snapshot_download
HF_TOKEN = ''  # isi jika perlu
MODEL_ID = 'aitf-komdigi/KomdigiITS-8B-DFK-TextClassification'

if not os.path.exists(os.path.join(MODEL_DIR, 'config.json')):
    print(f'Downloading {MODEL_ID} (~35 GB)...')
    snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_DIR,
        local_dir_use_symlinks=False, token=HF_TOKEN or None)
print('Files:', sorted(os.listdir(MODEL_DIR)))

In [ ]:
# Cell 4: Patch config.json dan tokenizer_config.json
import json as _json
cfg_path = os.path.join(MODEL_DIR, 'config.json')
with open(cfg_path) as f: cfg = _json.load(f)
changed = False
if cfg.get('model_type') in ('mistral3','ministral3'):
    cfg['model_type'] = 'mistral'
    cfg['architectures'] = ['MistralForCausalLM']
    changed = True
if 'generation_config' in cfg:
    del cfg['generation_config']
    changed = True
if changed:
    with open(cfg_path,'w') as f: _json.dump(cfg, f, indent=2)
    print('Patched config.json')

tok_path = os.path.join(MODEL_DIR, 'tokenizer_config.json')
with open(tok_path) as f: tok = _json.load(f)
if tok.get('tokenizer_class') == 'TokenizersBackend':
    tok['tokenizer_class'] = 'PreTrainedTokenizerFast'
    with open(tok_path,'w') as f: _json.dump(tok, f, indent=2)
    print('Patched tokenizer_config.json')
print('Patches done')

In [ ]:
# Cell 4.5: PATCH llama.cpp converter untuk handle prefix 'model.model.'
#
# ROOT CAUSE: model dilatih dengan PEFT/QLoRA yang menambah wrapper 'model.' layer,
# sehingga semua tensor punya prefix ganda 'model.model.' bukan 'model.'.
# Converter llama.cpp tidak mengenali 'model.model.embed_tokens.weight' dll.
# FIX: patch base.py untuk strip extra 'model.' sebelum tensor mapping.

patch_file = os.path.join(LLAMA_DIR, 'conversion', 'base.py')

with open(patch_file) as f:
    content = f.read()

# Verify belum di-patch
if 'model.model.' in content and 'startswith("model.model.")' not in content:
    # Tambahkan strip logic sebelum map_tensor_name dipanggil
    old = 'new_name = self.map_tensor_name(name)'
    new = (
        '# FIX: strip extra model. prefix from PEFT-trained models\n'
        '        if name.startswith("model.model."):\n'
        '            name = name[len("model."):]\n'
        '        new_name = self.map_tensor_name(name)'
    )
    if old in content:
        content = content.replace(old, new, 1)
        with open(patch_file, 'w') as f:
            f.write(content)
        print('Patched: base.py — strip model.model. prefix')
    else:
        # Backup: replace map_tensor_name call directly
        old2 = 'self.map_tensor_name(name)'
        new2 = 'self.map_tensor_name(name[6:] if name.startswith("model.model.") else name)'
        content = content.replace(old2, new2, 1)
        with open(patch_file, 'w') as f:
            f.write(content)
        print('Patched (backup): base.py')
else:
    print('Patch sudah ada atau tidak diperlukan')

# Verifikasi patch
with open(patch_file) as f:
    check = f.read()
if 'model.model.' in check or 'startswith' in check:
    print('Verifikasi: patch OK')
else:
    print('WARNING: Patch mungkin tidak teraplikasi, cek manual')

In [ ]:
# Cell 5: Convert ke GGUF f16
if os.path.exists(OUTPUT_F16): os.remove(OUTPUT_F16)

convert_script = os.path.join(LLAMA_DIR, 'convert_hf_to_gguf.py')
print(f'Converting {MODEL_DIR} -> {OUTPUT_F16}...')
print('(~10-15 menit)\n')

result = subprocess.run(
    [sys.executable, convert_script,
     MODEL_DIR, '--outfile', OUTPUT_F16, '--outtype', 'f16'],
    text=True, capture_output=True
)
if result.stdout: print('STDOUT:', result.stdout[-3000:])
if result.stderr: print('STDERR:', result.stderr[-3000:])

if result.returncode != 0:
    raise RuntimeError(f'GAGAL (exit {result.returncode})')

size_gb = os.path.getsize(OUTPUT_F16) / 1e9
assert size_gb >= 15, f'File terlalu kecil: {size_gb:.2f} GB'
print(f'f16 GGUF: {size_gb:.2f} GB ✓')

In [ ]:
# Cell 6: Quantize ke Q4_K_M
if os.path.exists(OUTPUT_Q4): os.remove(OUTPUT_Q4)

quantize_bin = os.path.join(LLAMA_DIR, 'build', 'bin', 'llama-quantize')
if not os.path.exists(quantize_bin):
    build_dir = os.path.join(LLAMA_DIR, 'build')
    os.makedirs(build_dir, exist_ok=True)
    subprocess.run(['cmake','-B',build_dir,LLAMA_DIR,
        '-DLLAMA_NATIVE=OFF','-DCMAKE_BUILD_TYPE=Release'], check=True)
    subprocess.run(['cmake','--build',build_dir,
        '--target','llama-quantize','-j4'], check=True)

subprocess.run([quantize_bin, OUTPUT_F16, OUTPUT_Q4, 'Q4_K_M'], check=True)

size_gb = os.path.getsize(OUTPUT_Q4) / 1e9
assert size_gb >= 4, f'File terlalu kecil: {size_gb:.2f} GB'
print(f'Q4_K_M GGUF: {size_gb:.2f} GB ✓')
os.remove(OUTPUT_F16)
print('f16 dihapus, hemat disk')

In [ ]:
# Cell 7: Upload ke HuggingFace Hub
from huggingface_hub import HfApi

HF_TOKEN      = ''  # WAJIB isi
GGUF_REPO     = 'ggapar/KomdigiITS-8B-DFK-GGUF'
GGUF_FILENAME = 'model-q4_k_m.gguf'

assert HF_TOKEN, 'Isi HF_TOKEN!'
size_gb = os.path.getsize(OUTPUT_Q4) / 1e9
assert size_gb >= 4, f'File corrupt ({size_gb:.2f} GB), jangan upload'

api = HfApi(token=HF_TOKEN)
api.create_repo(GGUF_REPO, repo_type='model', exist_ok=True)
print(f'Uploading {size_gb:.2f} GB...')
api.upload_file(
    path_or_fileobj=OUTPUT_Q4,
    path_in_repo=GGUF_FILENAME,
    repo_id=GGUF_REPO,
    repo_type='model',
    commit_message=f'Upload DFK GGUF Q4_K_M ({size_gb:.1f} GB) - fixed tensor prefix',
)
print(f'Upload selesai!')
print(f'https://huggingface.co/{GGUF_REPO}')